> 在线讲义：[第 44 章 · 推理请求的执行过程与资源需求：一条请求的时间线和账本](https://wikieden.github.io/ai-infra-from-constraints/lessons/44-inference-request.html) · 本书《从约束推导 AI Infra》

# 第 44 章 · 推理请求的执行过程与资源需求：一条请求的时间线和账本

配套讲义：仓库里 `ai-infra/notes/lessons/44-inference-request.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



RTX PRO 6000 上 Qwen3-8B 的请求时间线、KV 容量、TTFT / ITL 与吞吐取自 AII 第 8 章 §8.1。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
KiB, MiB, GiB = 2**10, 2**20, 2**30
K_TOKEN = 2 * 36 * 8 * 128 * 2  # 147,456 B

**函数 `request_time`**：(TTFT, 完整请求时间) 秒。

In [ ]:
def request_time(queue: float, prefill: float, itl: float, G: int = 256) -> tuple[float, float]:
    """(TTFT, 完整请求时间) 秒。"""
    ttft = queue + prefill
    return ttft, ttft + (G - 1) * itl

**函数 `kv_alloc_mib`**：生成完毕时按块分配的 KV（MiB）：输入 + G − 1 个回送的输出 token，按块上取整。

In [ ]:
def kv_alloc_mib(input_len: int, G: int = 256, block: int = 16) -> float:
    """生成完毕时按块分配的 KV（MiB）：输入 + G − 1 个回送的输出 token，按块上取整。"""
    tokens = math.ceil((input_len + G - 1) / block) * block
    return tokens * K_TOKEN / MiB

**函数 `fit`**：小工具。

In [ ]:
def fit(budget_mib: float, per_mib: float) -> int:
    return int(budget_mib // per_mib)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：一条 2K 短请求（排队 0.1 s、prefill 0.096 s、ITL 26.5 ms）")
ttft, total = request_time(0.1, 0.096, 0.0265)
print(f"  TTFT {ttft:.3f} s，完整 {total:.2f} s；插入一次 prefill 后 {total + 0.096:.2f} s（超 7 s）")

In [ ]:
print("\n第二步：KV 容量")
print(f"  每 token {K_TOKEN:,} B = {K_TOKEN // KiB} KiB；2K {2048 * K_TOKEN / MiB:.0f} MiB，8K {8192 * K_TOKEN / MiB:.0f} MiB")
for name, L in (("短对话 2K", 2048), ("长上下文 8K", 8192)):
    a = kv_alloc_mib(L)
    print(f"  {name}：生成完 {a:.0f} MiB，12 GiB 容纳 {fit(12 * 1024, a)} 条")
print(f"  只按 prefill 算会放 {fit(12 * 1024, 288)} 条短请求，生成完要 {42 * 324:,} MiB")
print(f"  16 条长请求只算输入就要 {16 * 1152 / 1024:.0f} GiB")

In [ ]:
print("\n第三步：吞吐")
print(f"  batch 16：每轮 27.44 ms 出 16 个 → {16 / 0.02744:.0f} token/s；batch 1：{1 / 0.02626:.0f} token/s")

## 3. 代码里才看得到的

1. 生成 256 个 token 只要把前 255 个送回模型：KV 长度是输入 + 255，不是 + 256。
2. 2048 + 255 = 2303，按每块 16 上取整成 2304。
3. 12 GiB 按 12,288 MiB 整除，余下的零头放不下一条完整请求。

## 4. 自己改一改

- 给 `kv_alloc_mib` 加块大小参数，比较 16、32、128 的浪费。
- 用 `request_time` 算 TTFT 小于 0.5 s、总时间小于 7 s 时允许的最长排队。
- 给 `fit` 加"16 条共享 864 MiB 前缀"的情况。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 长桌各吃各的，厨房每轮做每人的下一道；
> 第一道菜到手看排队加开火，之后看上菜间隔；插个别人的大单就超时；
> 车会变长，按进场时的长度卖票就停不下。